# 01. Data Cleaning & Preparation
**Project:** NTI Graduation Project <br>
**Dataset:** IBM HR Analytics Employee Attrition & Performance <br>
**Objective:** Audit, clean, prepare the raw dataset, and inspect/handle outliers for exploratory analysis.

### 1. Imports & Settings

In [1]:
import pandas as pd
import numpy as np

### 2. Load Raw Data

In [2]:
dataset = pd.read_csv("../data/raw_data.csv")
dataset.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


### 3. Initial Data Audit

In [3]:
print(f"Dataset Shape: {dataset.shape}")
print(f"Duplicates Count: {dataset.duplicated().sum()}")
print(f"Missing Values Total: {dataset.isna().sum().sum()}")

Dataset Shape: (1470, 35)
Duplicates Count: 0
Missing Values Total: 0


In [4]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

In [5]:
dataset.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,1470.0,36.923810,9.135373,18.0,30.00,36.0,43.00,60.0
DailyRate,1470.0,802.485714,403.509100,102.0,465.00,802.0,1157.00,1499.0
DistanceFromHome,1470.0,9.192517,8.106864,1.0,2.00,7.0,14.00,29.0
Education,1470.0,2.912925,1.024165,1.0,2.00,3.0,4.00,5.0
EmployeeCount,1470.0,1.000000,0.000000,1.0,1.00,1.0,1.00,1.0
EmployeeNumber,1470.0,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0
EnvironmentSatisfaction,1470.0,2.721769,1.093082,1.0,2.00,3.0,4.00,4.0
HourlyRate,1470.0,65.891156,20.329428,30.0,48.00,66.0,83.75,100.0
JobInvolvement,1470.0,2.729932,0.711561,1.0,2.00,3.0,3.00,4.0
JobLevel,1470.0,2.063946,1.106940,1.0,1.00,2.0,3.00,5.0


### 4. Remove Zero-Variance & Uninformative Columns

In [6]:
dataset.nunique()

Age                           43
Attrition                      2
BusinessTravel                 3
DailyRate                    886
Department                     3
DistanceFromHome              29
Education                      5
EducationField                 6
EmployeeCount                  1
EmployeeNumber              1470
EnvironmentSatisfaction        4
Gender                         2
HourlyRate                    71
JobInvolvement                 4
JobLevel                       5
JobRole                        9
JobSatisfaction                4
MaritalStatus                  3
MonthlyIncome               1349
MonthlyRate                 1427
NumCompaniesWorked            10
Over18                         1
OverTime                       2
PercentSalaryHike             15
PerformanceRating              2
RelationshipSatisfaction       4
StandardHours                  1
StockOptionLevel               4
TotalWorkingYears             40
TrainingTimesLastYear          7
WorkLifeBa

In [7]:
# Automatically identify columns with single unique value + drop EmployeeNumber (ID column)
constant_cols = [col for col in dataset.columns if dataset[col].nunique() <= 1]
cols_to_drop = constant_cols + ["EmployeeNumber"]

print(f"Columns to drop: {cols_to_drop}")

Columns to drop: ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']


In [8]:
dataset.drop(columns=[col for col in cols_to_drop if col in dataset.columns], inplace=True)
print(f"New Dataset Shape: {dataset.shape}")

New Dataset Shape: (1470, 31)


### 5. Outliers Management (IQR Method)

#### 5.1 Identify Rows Affected by Outliers

In [9]:
# Create a boolean mask for all rows containing at least one outlier
numeric_cols = dataset.select_dtypes(include=[np.number]).columns
continuous_cols = [col for col in numeric_cols if dataset[col].nunique() > 10]
outlier_mask = pd.Series(False, index=dataset.index)

for col in continuous_cols:
    Q1 = dataset[col].quantile(0.25)
    Q3 = dataset[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Update mask for rows with outliers in the current column
    outlier_mask |= (dataset[col] < lower_bound) | (dataset[col] > upper_bound)

# Filter dataset to get all actual outlier rows
actual_outliers_df = dataset[outlier_mask]

print(f"Total Rows with Outliers: {len(actual_outliers_df)}")
actual_outliers_df.head()

Total Rows with Outliers: 240


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
15,29,No,Travel_Rarely,1389,Research & Development,21,4,Life Sciences,2,Female,...,3,3,1,10,1,3,10,9,8,8
18,53,No,Travel_Rarely,1219,Sales,2,4,Life Sciences,1,Female,...,3,3,0,31,3,3,25,8,3,7
25,53,No,Travel_Rarely,1282,Research & Development,5,3,Other,3,Female,...,3,4,1,26,3,2,14,13,4,8
28,44,No,Travel_Rarely,477,Research & Development,7,4,Medical,1,Female,...,3,4,1,24,4,3,22,6,5,17
29,46,No,Travel_Rarely,705,Sales,2,4,Marketing,2,Female,...,3,4,0,22,2,2,2,2,2,1


#### 5.2 Impute Outliers with Median Values

In [10]:
# Replace outlier values in continuous columns with median
for col in continuous_cols:
    Q1 = dataset[col].quantile(0.25)
    Q3 = dataset[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Cast column to float to store float medians smoothly without type errors
    dataset[col] = dataset[col].astype(float)
    
    col_median = dataset[col].median()
    outlier_cond = (dataset[col] < lower_bound) | (dataset[col] > upper_bound)
    
    dataset.loc[outlier_cond, col] = col_median

print("All outliers successfully replaced with column medians!")

All outliers successfully replaced with column medians!


### 6. Categorical Data Type Optimization

In [11]:
categorical_cols = dataset.select_dtypes(include=['object', 'string', 'category']).columns
dataset[categorical_cols] = dataset[categorical_cols].astype('category')

dataset.dtypes

Age                          float64
Attrition                   category
BusinessTravel              category
DailyRate                    float64
Department                  category
DistanceFromHome             float64
Education                      int64
EducationField              category
EnvironmentSatisfaction        int64
Gender                      category
HourlyRate                   float64
JobInvolvement                 int64
JobLevel                       int64
JobRole                     category
JobSatisfaction                int64
MaritalStatus               category
MonthlyIncome                float64
MonthlyRate                  float64
NumCompaniesWorked             int64
OverTime                    category
PercentSalaryHike            float64
PerformanceRating              int64
RelationshipSatisfaction       int64
StockOptionLevel               int64
TotalWorkingYears            float64
TrainingTimesLastYear          int64
WorkLifeBalance                int64
Y

### 7. Export Cleaned Data

In [12]:
dataset.to_csv("../data/cleaned_data.csv", index=False)
print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
